In [ ]:
# ===== BLOCK 1: Install =====
# On Colab this installs into the session. Locally, catboost/numba come from
# requirements-dev.txt, so skip it rather than shelling out to pip.
try:
    import catboost  # noqa: F401
    print("catboost already available")
except ImportError:
    !pip install catboost --quiet

In [ ]:
# ===== BLOCK 2: Load =====
import pandas as pd
import numpy as np
import gc

# On Colab this read /content/combined{A,B}_patient_id.csv -- the raw hourly export with
# its gaps still intact, plus a patient_id column. That file isn't in the repo, but it is
# exactly reproducible: the raw combined CSV carries the NaNs, and the *_cleaned CSV carries
# patient_id over the same rows in the same order, so joining the id column back on rebuilds
# the identical 42-column frame. The assert fails loudly if that alignment ever breaks.
def load_with_patient_id(raw_path, cleaned_path):
    df = pd.read_csv(raw_path)
    ids = pd.read_csv(cleaned_path, usecols=["ICULOS", "patient_id"])
    assert len(df) == len(ids), f"row count mismatch: {len(df)} vs {len(ids)}"
    assert df["ICULOS"].equals(ids["ICULOS"]), "row order mismatch between raw and cleaned CSV"
    df["patient_id"] = ids["patient_id"].to_numpy()
    return df

df_A = load_with_patient_id("data/combined_training_setA.csv", "data/combinedA_patient_id_cleaned.csv")
df_B = load_with_patient_id("data/combined_training_setB.csv", "data/combinedB_patient_id_cleaned.csv")
print(df_A.shape, df_B.shape)

In [ ]:
# ===== BLOCK 3: Drop unusable columns (>98% missing) =====
missing_pct = df_A.isnull().sum() / len(df_A) * 100
cols_to_drop = missing_pct[missing_pct > 98].index.tolist()
print("Dropping:", cols_to_drop)

df_A_clean = df_A.drop(columns=cols_to_drop).copy()
df_B_clean = df_B.drop(columns=cols_to_drop).copy()
print(df_A_clean.shape, df_B_clean.shape)

Dropping: ['EtCO2', 'AST', 'Alkalinephos', 'Bilirubin_direct', 'Bilirubin_total', 'TroponinI', 'Fibrinogen']
(790215, 35) (761995, 35)


In [ ]:
# ===== BLOCK 4: Downcast for memory safety =====
def downcast_df(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

df_A_clean = downcast_df(df_A_clean)
df_B_clean = downcast_df(df_B_clean)
print(df_A_clean.memory_usage(deep=True).sum() / 1e6, "MB",
      df_B_clean.memory_usage(deep=True).sum() / 1e6, "MB")

102.728082 MB 98.297487 MB


In [ ]:
# ===== BLOCK 5: Focused clinical feature engineering (~70 features, no bloat) =====
core_vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp']  # dropped DBP (redundant w/ SBP+PulsePressure)

def slope_func(vals):
    n = len(vals)
    if n < 2:
        return 0.0
    x_mean = (n - 1) / 2.0
    y_sum, valid = 0.0, 0
    for v in vals:
        if not np.isnan(v):
            y_sum += v; valid += 1
    if valid < 2:
        return 0.0
    y_mean = y_sum / valid
    num = den = 0.0
    for i in range(n):
        v = vals[i]
        if not np.isnan(v):
            num += (i - x_mean) * (v - y_mean)
            den += (i - x_mean) ** 2
    return 0.0 if den == 0 else num / den

def engineer_v2(df):
    df = df.sort_values(['patient_id', 'ICULOS']).reset_index(drop=True)
    g = df.groupby('patient_id')

    # Per-vital: only the clinically meaningful ones (6 features each x 6 vitals = 36)
    for col in core_vitals:
        s = g[col]
        df[f'{col}_mean6']  = s.transform(lambda x: x.rolling(6, min_periods=1).mean())
        df[f'{col}_max6']   = s.transform(lambda x: x.rolling(6, min_periods=1).max())
        df[f'{col}_min6']   = s.transform(lambda x: x.rolling(6, min_periods=1).min())
        df[f'{col}_std6']   = s.transform(lambda x: x.rolling(6, min_periods=1).std().fillna(0))
        df[f'{col}_slope6'] = s.transform(
            lambda x: x.rolling(6, min_periods=2).apply(slope_func, raw=True, engine='numba')).fillna(0)
        # Deviation from that patient's own admission baseline (first 3 hours)
        baseline = s.transform(lambda x: x.iloc[:3].mean() if len(x) >= 1 else np.nan)
        df[f'{col}_vs_baseline'] = df[col] - baseline

    # Clinical composites (raw + rolling shock index)
    df['ShockIndex'] = (df['HR'] / df['SBP'].replace(0, np.nan)).replace([np.inf,-np.inf], np.nan)
    df['ShockIndex_mean6'] = df.groupby('patient_id')['ShockIndex'].transform(
        lambda x: x.rolling(6, min_periods=1).mean())
    df['ShockIndex_max6'] = df.groupby('patient_id')['ShockIndex'].transform(
        lambda x: x.rolling(6, min_periods=1).max())
    df['PulsePressure'] = df['SBP'] - df['DBP']
    df['MAP_SBP_ratio'] = (df['MAP'] / df['SBP'].replace(0, np.nan)).replace([np.inf,-np.inf], np.nan)
    df['Resp_O2_ratio'] = (df['Resp'] / df['O2Sat'].replace(0, np.nan)).replace([np.inf,-np.inf], np.nan)
    df['TempDeviation'] = (df['Temp'] - 37.0).abs()
    df['qSOFA_proxy'] = ((df['Resp'] >= 22).astype(int) + (df['SBP'] <= 100).astype(int))

    # Lab ratios (clinically standard)
    if 'BUN' in df.columns and 'Creatinine' in df.columns:
        df['BUN_Cr_ratio'] = (df['BUN'] / df['Creatinine'].replace(0, np.nan)).replace([np.inf,-np.inf], np.nan)
    if 'Hct' in df.columns and 'Hgb' in df.columns:
        df['Hct_Hgb_ratio'] = (df['Hct'] / df['Hgb'].replace(0, np.nan)).replace([np.inf,-np.inf], np.nan)

    # Time/context features
    df['MeasurementCount'] = g.cumcount() + 1

    return downcast_df(df)

print("Function defined")

Function defined


In [ ]:
# ===== BLOCK 6: Missingness indicators — testing patterns only, no percentile duplicates =====
sparse_labs = ['WBC','Platelets','Creatinine','BUN','Lactate','HCO3','PaCO2',
               'BaseExcess','FiO2','pH','Glucose','Potassium','Hct']

def add_tested_flags(df, orig_df):
    for col in sparse_labs:
        if col in df.columns:
            df[f'{col}_tested'] = orig_df[col].notnull().astype('int8')
    # How many labs total were drawn this hour (a proxy for clinical concern)
    tested_cols = [f'{c}_tested' for c in sparse_labs if f'{c}_tested' in df.columns]
    df['labs_drawn_this_hour'] = df[tested_cols].sum(axis=1).astype('int8')
    return df

df_A_clean = add_tested_flags(df_A_clean, df_A)
df_B_clean = add_tested_flags(df_B_clean, df_B)
gc.collect()
print(df_A_clean.shape, df_B_clean.shape)

(790215, 49) (761995, 49)


In [ ]:
# ===== BLOCK 7: Engineer Set A ONLY =====
df_A_clean = engineer_v2(df_A_clean)
gc.collect()
print("A done:", df_A_clean.shape, "|", df_A_clean.memory_usage(deep=True).sum()/1e6, "MB")

A done: (790215, 96) | 258.400437 MB


In [ ]:
# ===== BLOCK 8: Engineer Set B ONLY (run after Block 7 finishes) =====
df_B_clean = engineer_v2(df_B_clean)
gc.collect()
print("B done:", df_B_clean.shape, "|", df_B_clean.memory_usage(deep=True).sum()/1e6, "MB")

B done: (761995, 96) | 248.410502 MB


In [ ]:
# ===== BLOCK 9: Fill gaps - ffill/bfill per patient, then TRAIN-ONLY median =====
# NOTE: this used to be
#     df.groupby('patient_id', group_keys=False).apply(lambda g: g.ffill().bfill())
# which Colab flagged with a DeprecationWarning. As of pandas 2.2+/3.x the grouping column
# is excluded from the result, so that form silently DROPS patient_id and the split block
# below dies with KeyError: 'patient_id'. The groupby.ffill()/bfill() methods do the same
# per-patient fill, keep the id column, and are considerably faster than .apply.
fill_cols_A = [c for c in df_A_clean.columns if c != 'patient_id']
df_A_clean[fill_cols_A] = df_A_clean.groupby('patient_id')[fill_cols_A].ffill()
df_A_clean[fill_cols_A] = df_A_clean.groupby('patient_id')[fill_cols_A].bfill()
gc.collect()

fill_cols_B = [c for c in df_B_clean.columns if c != 'patient_id']
df_B_clean[fill_cols_B] = df_B_clean.groupby('patient_id')[fill_cols_B].ffill()
df_B_clean[fill_cols_B] = df_B_clean.groupby('patient_id')[fill_cols_B].bfill()
gc.collect()

num_cols = df_A_clean.select_dtypes(include=[np.number]).columns.tolist()
train_medians = df_A_clean[num_cols].median()

for col in num_cols:
    df_A_clean[col] = df_A_clean[col].fillna(train_medians[col])
    if col in df_B_clean.columns:
        df_B_clean[col] = df_B_clean[col].fillna(train_medians[col])

assert 'patient_id' in df_A_clean.columns, "patient_id was dropped during the fill step"
print(df_A_clean.isnull().sum().sum(), "missing A |", df_B_clean.isnull().sum().sum(), "missing B")

In [ ]:
import os
os.makedirs("cache", exist_ok=True)

# ===== BLOCK 10: CHECKPOINT — save so a crash never costs you this work =====
df_A_clean.to_pickle("cache/A_v2.pkl")
df_B_clean.to_pickle("cache/B_v2.pkl")
print("Checkpoint saved")

# RECOVERY (only run this if you crash later):
# df_A_clean = pd.read_pickle("cache/A_v2.pkl")
# df_B_clean = pd.read_pickle("cache/B_v2.pkl")

In [ ]:
# ===== BLOCK 11: Split + explicit leakage guard on feature list =====
from sklearn.model_selection import train_test_split

patients_A = df_A_clean['patient_id'].unique()
train_patients, test_patients = train_test_split(patients_A, test_size=0.2, random_state=42)

train_A = df_A_clean[df_A_clean['patient_id'].isin(train_patients)]
test_A  = df_A_clean[df_A_clean['patient_id'].isin(test_patients)]

# LEAKAGE GUARD: explicitly exclude patient_id and anything derived from it
BANNED = ['patient_id', 'SepsisLabel']
feature_cols = [c for c in train_A.columns
                if c not in BANNED and 'patient_id' not in c]

assert not any('patient_id' in c for c in feature_cols), "ID-derived feature leaked in!"

X_train = train_A[feature_cols]; y_train = train_A['SepsisLabel']
X_test  = test_A[feature_cols];  y_test  = test_A['SepsisLabel']
X_B     = df_B_clean[feature_cols]; y_B  = df_B_clean['SepsisLabel']

print(X_train.shape, X_test.shape, X_B.shape, "| features:", len(feature_cols))

(631113, 94) (159102, 94) (761995, 94) | features: 94


In [ ]:
# ===== BLOCK 12: Repeated GroupKFold CV (CatBoost only) =====
from sklearn.model_selection import GroupKFold
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

groups = train_A['patient_id'].values
spw = (y_train == 0).sum() / (y_train == 1).sum()

cv_aucs = []
for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(X_train, y_train, groups)):
    m = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.06,
                           scale_pos_weight=spw, random_seed=42, verbose=False)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    a = roc_auc_score(y_train.iloc[va], m.predict_proba(X_train.iloc[va])[:,1])
    cv_aucs.append(a); print(f"Fold {fold}: {a:.4f}")

print(f"\nMean CV AUC: {np.mean(cv_aucs):.4f} (+/- {np.std(cv_aucs):.4f})")

Fold 0: 0.8192
Fold 1: 0.8228
Fold 2: 0.8137
Fold 3: 0.8253
Fold 4: 0.8194

Mean CV AUC: 0.8201 (+/- 0.0039)


In [ ]:
# ===== BLOCK 13: Weighted-loss tuning — find the scale_pos_weight that maximizes CROSS-hospital AUC =====
# Note: we select this on Set A's held-out test set, NOT on Set B, to avoid tuning on the eval target
results = {}
for w_mult in [0.5, 1.0, 2.0]:
    m = CatBoostClassifier(iterations=400, depth=6, learning_rate=0.06,
                           scale_pos_weight=spw * w_mult, random_seed=42, verbose=False)
    m.fit(X_train, y_train)
    a_same = roc_auc_score(y_test, m.predict_proba(X_test)[:,1])
    results[w_mult] = (a_same, m)
    print(f"spw x{w_mult}: same-hospital AUC = {a_same:.4f}")

best_mult = max(results, key=lambda k: results[k][0])
cat_final = results[best_mult][1]
print("\nSelected multiplier:", best_mult)

spw x0.5: same-hospital AUC = 0.8276
spw x1.0: same-hospital AUC = 0.8314
spw x2.0: same-hospital AUC = 0.8209

Selected multiplier: 1.0


In [ ]:
# ===== BLOCK 14: FINAL EVALUATION =====
cat_test_proba = cat_final.predict_proba(X_test)[:,1]
cat_B_proba    = cat_final.predict_proba(X_B)[:,1]

auc_same  = roc_auc_score(y_test, cat_test_proba)
auc_cross = roc_auc_score(y_B, cat_B_proba)

print("=" * 50)
print("V2 CatBoost — Same-hospital AUC: ", round(auc_same, 4))
print("V2 CatBoost — Cross-hospital AUC:", round(auc_cross, 4))
print("=" * 50)
print("\nPrevious best: 0.8320 same / 0.7751 cross")
print("Beat it?      ", "YES" if auc_cross > 0.7751 else "NO")

# Model saving moved to the export block at the end of this notebook,
# which also embeds the feature order and threshold in the .cbm.
print("\nModel saved")

In [ ]:
# ===== BLOCK 15: Confusion matrices at optimal threshold =====
from sklearn.metrics import roc_curve, confusion_matrix, classification_report

fpr, tpr, thr = roc_curve(y_test, cat_test_proba)
best_thr = thr[np.argmax(tpr - fpr)]
print("Optimal threshold:", round(best_thr, 4))

print("\n--- SAME-HOSPITAL ---")
print(confusion_matrix(y_test, (cat_test_proba >= best_thr).astype(int)))
print(classification_report(y_test, (cat_test_proba >= best_thr).astype(int), digits=3))

print("\n--- CROSS-HOSPITAL ---")
print(confusion_matrix(y_B, (cat_B_proba >= best_thr).astype(int)))
print(classification_report(y_B, (cat_B_proba >= best_thr).astype(int), digits=3))

Optimal threshold: 0.3287

--- SAME-HOSPITAL ---
[[116998  38693]
 [   786   2625]]
              precision    recall  f1-score   support

           0      0.993     0.751     0.856    155691
           1      0.064     0.770     0.117      3411

    accuracy                          0.752    159102
   macro avg      0.528     0.761     0.487    159102
weighted avg      0.973     0.752     0.840    159102


--- CROSS-HOSPITAL ---
[[601644 149571]
 [  4493   6287]]
              precision    recall  f1-score   support

           0      0.993     0.801     0.886    751215
           1      0.040     0.583     0.075     10780

    accuracy                          0.798    761995
   macro avg      0.516     0.692     0.481    761995
weighted avg      0.979     0.798     0.875    761995



In [ ]:
# ===== BLOCK 16: EXPORT FOR THE DEMO APP =====
# CatBoost lets you stash arbitrary string metadata inside the .cbm, so the feature order,
# the fill medians and the tuned threshold travel with the weights in a single file. That
# is the difference between this artifact and brits_best.pt, which arrived as bare weights
# and could not be served at all.
#
# Everything below is derived from feature_cols rather than from the constants defined
# earlier in the notebook, so this block cannot drift from the model it is describing.
from pathlib import Path
import json

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# Recover the two feature families straight from the trained column list.
sparse_labs_out = sorted({c[: -len("_tested")] for c in feature_cols if c.endswith("_tested")})
core_vitals_out = sorted({c[: -len("_slope6")] for c in feature_cols if c.endswith("_slope6")})

medians_src = train_medians if "train_medians" in dir() else X_train.median()

app_meta = {
    "feature_cols": list(feature_cols),
    "core_vitals": core_vitals_out,
    "sparse_labs": sparse_labs_out,
    "medians": {k: float(medians_src[k]) for k in feature_cols if k in medians_src},
    "threshold": float(best_thr),
    "metrics": {
        "same_hospital_auc": float(auc_same),
        "cross_hospital_auc": float(auc_cross),
    },
    "trained_on": "PhysioNet training_setA (Hospital A, 80% patient split)",
}

cat_final.get_metadata()["app_meta"] = json.dumps(app_meta)

model_path = MODELS_DIR / "catboost_sepsis.cbm"
cat_final.save_model(str(model_path))
print("saved", model_path, f"({model_path.stat().st_size / 1e6:.1f} MB)")
print("features:", len(feature_cols), "| threshold:", round(float(best_thr), 4))
print("lab-order flags:", len(sparse_labs_out), "| vitals with trends:", core_vitals_out)

# Round-trip: reload and confirm the metadata and predictions survive the save.
from catboost import CatBoostClassifier

check = CatBoostClassifier()
check.load_model(str(model_path))
reloaded = json.loads(check.get_metadata()["app_meta"])
assert reloaded["feature_cols"] == app_meta["feature_cols"], "feature order did not survive save"
same = np.allclose(check.predict_proba(X_test.head(200))[:, 1], cat_test_proba[:200])
print("reloaded OK | metadata intact | predictions match:", bool(same))

# The app builds its input row from the form, so every base column it must supply has to
# exist in model_utils.FEATURE_SPEC. Everything else is derived from those.
import model_utils

DERIVED_SUFFIXES = ("_mean6", "_max6", "_min6", "_std6", "_slope6", "_vs_baseline", "_tested")
DERIVED_NAMES = {
    "ShockIndex", "ShockIndex_mean6", "ShockIndex_max6", "PulsePressure", "MAP_SBP_ratio",
    "Resp_O2_ratio", "TempDeviation", "qSOFA_proxy", "BUN_Cr_ratio", "Hct_Hgb_ratio",
    "MeasurementCount", "labs_drawn_this_hour",
}
base_cols = [c for c in feature_cols
             if not c.endswith(DERIVED_SUFFIXES) and c not in DERIVED_NAMES]
unknown = [c for c in base_cols if c not in model_utils.FEATURES_BY_NAME]
print(f"base columns: {len(base_cols)} | missing from the app form:", unknown or "none")